# CompEcon vs hetmacro — Visual Comparison

This notebook provides visual comparisons between CompEcon and hetmacro for key foundational tools. It focuses on:

- Quadrature accuracy and node/weight behavior
- Chebyshev approximation quality
- Grid orientation differences

The goal is to make differences and compatibility clear at a glance.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from compecon import quad as cquad
from compecon import basisChebyshev
from compecon import tools as ctools

from hetmacro import quadrature as hquad
from hetmacro import interpolation as hinterp
from hetmacro import grids as hgrids

plt.rcParams.update({"figure.figsize": (8, 4), "axes.grid": True})

print("Imports loaded.")

## 1) Quadrature comparison (Legendre and Chebyshev)

We compare:
- **Gauss‑Legendre** accuracy for integrating a smooth function on a bounded interval.
- **Chebyshev rules**: hetmacro Gauss‑Chebyshev vs CompEcon Clenshaw‑Curtis.

In [ ]:
def quad_expectation(nodes, weights, f):
    return np.sum(weights * f(nodes))

# Test function and exact integral on [-1,1]
# f(x) = exp(x), integral = e - 1/e
f = np.exp
exact = np.e - 1 / np.e

ns = [3, 5, 7, 9, 11]
err_h = []
err_c = []

for n in ns:
    xh, wh = hquad.qnwlege(n, -1, 1)
    xc, wc = cquad.qnwlege(n, -1, 1)
    err_h.append(abs(quad_expectation(xh, wh, f) - exact))
    err_c.append(abs(quad_expectation(xc, wc, f) - exact))

plt.figure(figsize=(7, 4))
plt.semilogy(ns, err_h, "o-", label="hetmacro qnwlege")
plt.semilogy(ns, err_c, "s--", label="CompEcon qnwlege")
plt.xlabel("n")
plt.ylabel("abs error")
plt.title("Gauss-Legendre accuracy on exp(x), [-1,1]")
plt.legend()
plt.show()

In [ ]:
n = 9

# CompEcon Chebyshev = Clenshaw-Curtis weights on Chebyshev nodes
xc, wc = cquad.qnwcheb(n, -1, 1)

# hetmacro: Gauss-Chebyshev (weighted) vs Clenshaw-Curtis (CompEcon-compatible)
xg, wg = hquad.qnwcheb(n, -1, 1, kind="gauss")
xl, wl = hquad.qnwcheb(n, -1, 1, kind="clenshaw_curtis")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Nodes comparison
axes[0].plot(xc, np.zeros_like(xc), "o", label="CompEcon (Clenshaw-Curtis)")
axes[0].plot(xl, np.zeros_like(xl), "s", label="hetmacro clenshaw_curtis")
axes[0].plot(xg, np.zeros_like(xg), "x", label="hetmacro gauss")
axes[0].set_title("Chebyshev nodes")
axes[0].legend()

# Weights comparison
axes[1].plot(xc, wc, "o", label="CompEcon (Clenshaw-Curtis)")
axes[1].plot(xl, wl, "s", label="hetmacro clenshaw_curtis")
axes[1].plot(xg, wg, "x", label="hetmacro gauss")
axes[1].set_title("Chebyshev weights")
axes[1].legend()

plt.suptitle("Chebyshev rules: CompEcon vs hetmacro", y=1.02)
plt.tight_layout()
plt.show()

print("Sum of weights:")
print("CompEcon (clenshaw):", wc.sum())
print("hetmacro clenshaw:   ", wl.sum())
print("hetmacro gauss:      ", wg.sum())

## 2) Chebyshev approximation (function interpolation)

We approximate the same function with Chebyshev bases in both libraries and compare the fit.

In [ ]:
a, b = -1.0, 1.0
n = 9
x_plot = np.linspace(a, b, 400)

# hetmacro Chebyshev approximation
coef = hinterp.cheb_coef(np.exp, n=n, a=a, b=b)
fh = hinterp.cheb_eval(coef, x_plot, a=a, b=b)

# CompEcon Chebyshev approximation (BasisChebyshev)
cb = basisChebyshev.BasisChebyshev(n, a, b)
cb.y = np.exp(cb.x)
cb.update_c()
Phi = cb.Phi(x_plot)
fc = (cb.c @ Phi.T).ravel()

# Plot
plt.figure(figsize=(8, 4))
plt.plot(x_plot, np.exp(x_plot), label="true", linewidth=2)
plt.plot(x_plot, fh, "--", label="hetmacro cheb")
plt.plot(x_plot, fc, ":", label="CompEcon cheb")
plt.title("Chebyshev approximation of exp(x)")
plt.legend()
plt.show()

# Error comparison
err_h = np.max(np.abs(fh - np.exp(x_plot)))
err_c = np.max(np.abs(fc - np.exp(x_plot)))
print("Max error hetmacro:", err_h)
print("Max error CompEcon:", err_c)

## 3) Grid orientation difference

Both libraries generate the same Cartesian product points, but the output layout differs:
- **CompEcon**: shape `(d, N)`
- **hetmacro**: shape `(N, d)`

In [ ]:
x1 = np.array([1, 2, 3])
x2 = np.array([4, 5])

cg = ctools.gridmake(x1, x2)
hg = hgrids.gridmake(x1, x2)

print("CompEcon gridmake shape:", cg.shape)
print("hetmacro gridmake shape:", hg.shape)

print("CompEcon (d,N)\n", cg)
print("hetmacro (N,d)\n", hg)

# They contain the same points, just transposed
print("Points equal after transpose:", np.allclose(cg.T, hg))

## Summary

- Quadrature: Legendre rules match closely; Chebyshev differs by default because CompEcon uses Clenshaw–Curtis weights. Use `qnwcheb(..., kind="clenshaw_curtis")` for CompEcon‑compatible results.
- Chebyshev approximation: both libraries produce similar fits for smooth functions.
- Gridmake: same points, different output orientation.